# 00 - Dataset browser

**Question this notebook answers:** which sources go into the study?

Nothing here downloads or processes a full dataset. It reads the source manifest,
probes each entry cheaply, and shows you thumbnails so you can decide.

| You see | You decide |
|---|---|
| Source table, licence, duration | Is this source usable at all? |
| Thumbnail grid per source | Is it side view? One cow? Usable framing? |

Output when you commit: `data/sources_selected.csv`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 60)
print("project root:", PROJECT_ROOT)

## Run configuration

Everything downstream honours these three flags.

In [ ]:
PREVIEW_ONLY = True     # True = look only, never write
MAX_SAMPLES = 20        # thumbnails per source in preview
SAVE_OUTPUTS = False    # must ALSO be True before any file is written

SOURCES_CSV = PROJECT_ROOT / "data" / "sources.csv"
SELECTED_CSV = PROJECT_ROOT / "data" / "sources_selected.csv"
THUMBNAIL_FPS = 0.2     # one frame every 5 seconds while browsing

In [ ]:
def guard_save(what: str) -> bool:
    """Refuse to write anything while the notebook is in preview mode."""
    if PREVIEW_ONLY or not SAVE_OUTPUTS:
        print(f"PREVIEW MODE - not writing {what}. Set PREVIEW_ONLY=False and SAVE_OUTPUTS=True to commit.")
        return False
    return True

## 1. Source manifest

`data/sources.csv` is written by hand (copy `sources.example.csv`). Sources that
were downloaded through `scripts/01_collect.py` land in `sources_resolved.csv`
with a `resolved_path` column; either file works here.

In [ ]:
from cowarch.frames import probe_source

if not SOURCES_CSV.exists():
    raise FileNotFoundError(
        f"{SOURCES_CSV} is missing. Copy sources.example.csv to data/sources.csv "
        "and fill in one row per source, including license_status."
    )

from cowarch.sources import normalize_source_schema

sources = normalize_source_schema(
    pd.read_csv(SOURCES_CSV, keep_default_na=False, comment="#")
)
sources["path"] = [
    row.get("resolved_path", "") or row.get("local_path", "") for _, row in sources.iterrows()
]
def probe_entry(path_value, kind, license_status):
    if license_status != "approved":
        return {"exists": False, "kind": kind, "path": str(path_value), "error": "blocked by license_status"}
    if not str(path_value).strip():
        return {"exists": False, "kind": kind, "path": "", "error": "not resolved"}
    path = Path(path_value)
    return probe_source(PROJECT_ROOT / path if not path.is_absolute() else path, kind)

probe = pd.DataFrame(
    [probe_entry(p, k, status) for p, k, status in zip(
        sources["path"], sources["kind"], sources["license_status"]
    )]
).drop(columns=["kind"], errors="ignore")
source_columns = ["source_id", "kind", "license_status", "license_name", "license_url"]
overview = pd.concat([sources[source_columns], probe], axis=1)
overview

### Licence check

Only `license_status=approved` is a green light. `restricted`, `unresolved`,
and legacy placeholders (`check-before-use`, `unknown`, blank) are blocked by
`scripts/01_collect.py` before any path is resolved or any download starts.

In [ ]:
blocked = overview[overview["license_status"].ne("approved")]
if len(blocked):
    print(f"{len(blocked)} source(s) are blocked by the license gate:")
    display(blocked[["source_id", "license_status", "license_name", "license_url", "path"]])
else:
    print("every source is explicitly approved")

missing = overview[overview["license_status"].eq("approved") & ~overview["exists"]]
if len(missing):
    print("\nmissing on disk - run scripts/01_collect.py or fix the path:")
    display(missing[["source_id", "path"]])

## 2. Thumbnails

Look for: side view, a single cow in frame, the whole body visible, the back not
occluded by rails or another animal. A source that never shows a clean lateral
pass is worth dropping now rather than filtering frame by frame later.

In [ ]:
from cowarch.frames import sample_frames
from cowarch.viz import thumbnail_grid

def browse(source_id: str, fps: float = THUMBNAIL_FPS, n: int = MAX_SAMPLES):
    row = overview.loc[overview["source_id"] == source_id].iloc[0]
    path = Path(row["path"])
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    frames = sample_frames(path, row["kind"], fps, n)
    if not frames:
        print(f"{source_id}: no readable frames")
        return
    fig = thumbnail_grid(
        [frame for _, frame, _ in frames],
        [f"#{idx}" for idx, _, _ in frames],
        columns=5,
    )
    fig.suptitle(f"{source_id} - {len(frames)} sampled frames @ {fps} fps", y=1.01)
    plt.show()

previewable = overview["exists"] & overview["license_status"].eq("approved")
for source_id in overview.loc[previewable, "source_id"]:
    browse(source_id)

## 3. Your decision

Edit `KEEP` by hand. This is the point of the notebook: the selection is yours,
not a threshold's.

In [ ]:
KEEP = [
    # "source_id_1",
    # "source_id_2",
]

DROP_REASONS = {
    # "source_id_3": "front view only",
}

if not KEEP:
    print("KEEP is empty - fill it in from the thumbnails above before committing.")
else:
    selected = sources[sources["source_id"].isin(KEEP)].copy()
    unapproved = selected[selected["license_status"].ne("approved")]
    if len(unapproved):
        raise ValueError(
            "KEEP contains sources that are not approved: "
            + ", ".join(unapproved["source_id"].astype(str))
        )
    print(f"{len(selected)} of {len(sources)} sources selected")
    display(selected[["source_id", "kind", "license_status", "license_name", "path"]])
    if len(selected) < 8:
        print(
            f"\nWARNING: {len(selected)} groups. The plan asks for 8-10 independent "
            "groups so that a group-wise split can still put both classes in every "
            "split. Fewer groups makes a degenerate val/test split likely."
        )

In [ ]:
if KEEP and guard_save(str(SELECTED_CSV)):
    SELECTED_CSV.parent.mkdir(parents=True, exist_ok=True)
    selected.drop(columns=["path"]).to_csv(SELECTED_CSV, index=False)
    print("wrote", SELECTED_CSV)
    print("\nNext: 01_detection_segmentation_inspector.ipynb")